# 예측해보기

> 파이썬 16강 · 시계열과 예측 · 마지막 강의

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [예측해보기](https://mioon1402.github.io/timeseriesdata/python/p16-forecast.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.** 예시 데이터를 내려받습니다.

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv

# 표를 글자로 찍을 때 한글 열이 어긋나지 않게 (한글을 두 칸으로 계산)
import pandas as pd
pd.set_option("display.unicode.east_asian_width", True)

# 그래프 한글 깨짐 방지
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # noqa: F401

print('준비 완료')

## 1. 학습/검증 분리

**16-1. 시간 순서대로 자르기**

In [ ]:
import pandas as pd

df = pd.read_csv("cafe_sales.csv", parse_dates=["date"])
s = df.set_index("date").asfreq("D")["sales"].interpolate()

H = 28                       # 예측 기간: 4주
학습, 검증 = s[:-H], s[-H:]

print(f"학습  {len(학습):3d}일  {학습.index[0].date()} ~ {학습.index[-1].date()}")
print(f"검증  {len(검증):3d}일  {검증.index[0].date()} ~ {검증.index[-1].date()}")

## 2. 평가 지표

**16-2. 평가 함수 만들기**

In [ ]:
import numpy as np

기록 = {}   # 모델 이름 → 점수. 같은 이름을 다시 평가하면 덮어쓴다

def 평가(이름, 예측):
    실제 = 검증.values
    예측 = np.asarray(예측, dtype=float)
    mae  = np.mean(np.abs(실제 - 예측))
    rmse = np.sqrt(np.mean((실제 - 예측) ** 2))
    mape = np.mean(np.abs((실제 - 예측) / 실제)) * 100
    기록[이름] = {"MAE": round(mae), "RMSE": round(rmse), "MAPE(%)": round(mape, 1)}
    return mape

print("평가 함수 준비 완료 — 점수는 기록에 쌓입니다")

## 3. 기준선부터 만들기

**16-3. 세 가지 기준선**

In [ ]:
결과 = {}

# ① 마지막 값을 그대로 (나이브)
결과["① 마지막값"] = 평가("① 마지막값 그대로", np.repeat(학습.iloc[-1], H))

# ② 최근 28일 평균
결과["② 최근28일 평균"] = 평가("② 최근 28일 평균", np.repeat(학습.iloc[-28:].mean(), H))

# ③ 지난주 같은 요일 (계절성 나이브)
결과["③ 지난주 같은요일"] = 평가("③ 지난주 같은 요일", 학습.iloc[-7:].values.tolist() * 4)

pd.DataFrame(기록).T   # 마지막 줄에 값만 두면 표로 보여준다

## 4. 지수평활과 SARIMA

**16-4. 홀트윈터스 지수평활**

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

모델 = ExponentialSmoothing(
    학습,
    trend="add",              # 추세 있음
    damped_trend=True,        # 추세를 무한정 연장하지 않음
    seasonal="add",           # 계절성 있음
    seasonal_periods=7,       # 주기 7일
).fit()

결과["④ 홀트윈터스"] = 평가("④ 홀트윈터스(주간)", 모델.forecast(H).values)

pd.DataFrame(기록).T

**16-5. SARIMA**

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

사리마 = ARIMA(
    학습,
    order=(1, 1, 1),                  # (p, d, q) — 비계절 부분
    seasonal_order=(1, 1, 1, 7),      # (P, D, Q, s) — 계절 부분, 주기 7
).fit()

결과["⑤ SARIMA"] = 평가("⑤ SARIMA(1,1,1)(1,1,1,7)", 사리마.forecast(H).values)

pd.DataFrame(기록).T

## 5. 결과 — 단순한 게 이겼습니다

**16-6. 순위표**

In [ ]:
print("MAPE 낮은 순\n")
for 이름, 값 in sorted(결과.items(), key=lambda x: x[1]):
    막대 = "█" * int(값 * 2)
    print(f"  {값:5.1f}%  {막대}  {이름}")

**16-7. 예측을 그림으로**

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10.5, 4))

최근학습 = 학습.iloc[-60:]
ax.plot(최근학습.index, 최근학습 / 10000, color="#94a3b8", lw=1, label="학습 데이터")
ax.plot(검증.index, 검증 / 10000, color="black", lw=1.8, label="실제")
ax.plot(검증.index, 사리마.forecast(H).values / 10000,
        color="#2563eb", lw=1.8, ls="--", label="SARIMA 예측")
ax.plot(검증.index, np.repeat(학습.iloc[-28:].mean(), H) / 10000,
        color="#dc2626", lw=1.8, ls=":", label="최근 28일 평균")

ax.axvline(검증.index[0], color="gray", ls="--", lw=1)
ax.set_ylabel("매출(만원)")
ax.set_title("예측 vs 실제 — 단순 평균이 요일 변동을 못 따라가지만 평균적으론 더 정확",
             fontweight="bold", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

## 6. 예측구간 — 점이 아니라 범위로

**16-8. 95% 예측구간**

In [ ]:
예보 = 사리마.get_forecast(H)
구간 = 예보.conf_int()

표 = pd.DataFrame({
    "예측": 예보.predicted_mean.values / 10000,
    "하한": 구간.iloc[:, 0].values / 10000,
    "상한": 구간.iloc[:, 1].values / 10000,
    "실제": 검증.values / 10000,
}, index=검증.index).round(1)

print(표.head(7).to_string())

포함 = ((표["실제"] >= 표["하한"]) & (표["실제"] <= 표["상한"])).sum()
print(f"\n95% 구간이 실제를 포함: {포함}/{H}일 ({포함/H*100:.0f}%)")

**16-9. 예측구간 그리기**

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4))

ax.plot(학습.iloc[-45:].index, 학습.iloc[-45:] / 10000, color="#94a3b8", lw=1)
ax.plot(검증.index, 검증 / 10000, color="black", lw=1.8, label="실제")
ax.plot(검증.index, 표["예측"], color="#2563eb", lw=1.8, label="예측")
ax.fill_between(검증.index, 표["하한"], 표["상한"],
                alpha=0.2, color="#2563eb", label="95% 예측구간")

ax.set_ylabel("매출(만원)")
ax.set_title("정직한 예측은 폭이 넓다", fontweight="bold", loc="left")
ax.legend(frameon=False, fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

## 7. 실무 체크리스트

**16-10. 여러 시점에서 검증하기 (롤링 검증)**

In [ ]:
# 한 번의 검증은 운일 수 있다. 시점을 옮겨가며 여러 번 평가한다.
점수 = []
for 끝 in [28, 56, 84, 112, 140]:
    tr = s[:-끝]
    te = s[-끝:-끝+H] if 끝 > H else s[-끝:]
    if len(te) < H:
        continue
    예측 = np.repeat(tr.iloc[-28:].mean(), len(te))
    mape = np.mean(np.abs((te.values - 예측) / te.values)) * 100
    점수.append(mape)
    print(f"  {te.index[0].date()} 부터 {len(te)}일   MAPE {mape:5.1f}%")

print(f"\n평균 MAPE {np.mean(점수):.1f}%   표준편차 {np.std(점수):.1f}%p")

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 예측 기간 H 를 7일로 바꿔서 다시 비교해보세요.
#        짧은 예측에서는 어떤 방법이 이기나요?


# 문제 2. 방문객(visitors)을 예측해보세요. 매출보다 쉬운가요, 어려운가요?


# 문제 3. SARIMA 의 order 를 (2,1,2) 로 바꿔보세요. 나아지나요?

**모범 답안**

In [ ]:
def 실험(시리즈, H):
    tr, te = 시리즈[:-H], 시리즈[-H:]
    def m(p):
        p = np.asarray(p, dtype=float)
        return np.mean(np.abs((te.values - p) / te.values)) * 100
    out = {
        "마지막값":      m(np.repeat(tr.iloc[-1], H)),
        "최근28일 평균": m(np.repeat(tr.iloc[-28:].mean(), H)),
        "지난주 요일":   m((tr.iloc[-7:].values.tolist() * (H // 7 + 1))[:H]),
    }
    try:
        a = ARIMA(tr, order=(1,1,1), seasonal_order=(1,1,1,7)).fit()
        out["SARIMA"] = m(a.forecast(H).values)
    except Exception as e:
        out["SARIMA"] = float("nan")
    return out

# 문제 1
print("[매출 · 7일 예측]")
for k, v in sorted(실험(s, 7).items(), key=lambda x: x[1]):
    print(f"  {v:5.1f}%  {k}")

# 문제 2
v_series = df.set_index("date").asfreq("D")["visitors"].interpolate()
print("\n[방문객 · 28일 예측]")
for k, val in sorted(실험(v_series, 28).items(), key=lambda x: x[1]):
    print(f"  {val:5.1f}%  {k}")

# 문제 3
print("\n[SARIMA order 비교 · 28일]")
for order in [(1,1,1), (2,1,2), (0,1,1)]:
    try:
        a = ARIMA(학습, order=order, seasonal_order=(1,1,1,7)).fit()
        p = a.forecast(H).values
        mape = np.mean(np.abs((검증.values - p) / 검증.values)) * 100
        print(f"  order={order}  MAPE {mape:.1f}%")
    except Exception as e:
        print(f"  order={order}  실패: {type(e).__name__}")

---

전체 강의 목록 → [눈으로 보는 통계](https://mioon1402.github.io/timeseriesdata/)